In [1]:
import os

# Clear proxy environment variables to ensure requests go to Google, not localhost
os.environ.pop('HTTP_PROXY', None)
os.environ.pop('HTTPS_PROXY', None)
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)

print("Proxies cleared.")

Proxies cleared.


In [3]:
import pandas as pd

# Load the dataset (adjust filename if necessary)
df = pd.read_csv("yelp.csv")

# Ensure we have the necessary columns (usually 'text' and 'stars')
# Filter for only relevant columns to save memory
df = df[['text', 'stars']]

# Sample 200 rows for the experiment
sampled_df = df.sample(n=200, random_state=42).reset_index(drop=True)

print(f"Dataset shape: {sampled_df.shape}")
sampled_df.head()

Dataset shape: (200, 2)


,text,stars
0,We got here around midnight last Friday... the...,4
1,Brought a friend from Louisiana here. She say...,5
2,"Every friday, my dad and I eat here. We order ...",3
3,"My husband and I were really, really disappoin...",1
4,Love this place! Was in phoenix 3 weeks for w...,5


In [ ]:
import os
from openai import OpenAI

# 1. Set your key directly here
my_groq_key = "YOUR_API_KEY"  # <--- PASTE YOUR ACTUAL KEY INSIDE THE QUOTES

# 2. Initialize the client using this variable
client = OpenAI(
    api_key=my_groq_key,
    base_url="https://api.groq.com/openai/v1",
)

print("Client initialized successfully!")

Client initialized successfully!


In [ ]:
import os
import json
import time
import pandas as pd
from openai import OpenAI

# --- 1. CONFIGURATION ---
# Initialize the client by passing the key string DIRECTLY
client = OpenAI(
    api_key="YOUR_API_KEY", # <--- Removed os.environ.get
    base_url="https://api.groq.com/openai/v1",
)

# Define the model
# Note: Ensure "openai/gpt-oss-20b" is the correct ID for Groq.
# If it fails, switch to "llama3-8b-8192" or "mixtral-8x7b-32768"
GROQ_MODEL = "llama-3.1-8b-instant"

# --- 2. EXECUTION ENGINE ---
def run_experiment(df, prompt_func, approach_name):
    print(f"--- Starting Experiment: {approach_name} ---")
    results = []
    consecutive_errors = 0

    for index, row in df.iterrows():
        # Safety break
        if consecutive_errors >= 5:
            print("Stopping due to repeated API errors.")
            break

        try:
            # 1. Prepare Prompt
            user_content = prompt_func(row['text'])

            # 2. Call Groq API
            response = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    # System prompt helps enforce JSON mode behavior
                    {"role": "system", "content": "You are a helpful assistant that outputs strictly in JSON format."},
                    {"role": "user", "content": user_content}
                ],
                # Important: specific feature to guarantee JSON validity
                response_format={"type": "json_object"},
                temperature=0.1
            )

            # 3. Parse Response
            response_text = response.choices[0].message.content
            data = json.loads(response_text)

            results.append({
                "approach": approach_name,
                "actual_stars": row['stars'],
                "predicted_stars": data.get("predicted_stars"),
                "explanation": data.get("explanation"),
                "json_valid": True
            })
            consecutive_errors = 0

        except Exception as e:
            consecutive_errors += 1
            print(f"Row {index} Error: {e}")
            results.append({
                "approach": approach_name,
                "actual_stars": row['stars'],
                "predicted_stars": None,
                "explanation": str(e),
                "json_valid": False
            })

        time.sleep(0.5)

        if (index + 1) % 50 == 0:
            print(f"Processed {index + 1} reviews...")

    return pd.DataFrame(results)

In [6]:
def get_zero_shot_prompt(review_text):
    return f"""
    Analyze the sentiment of this Yelp review and predict the star rating (1-5).

    Review: "{review_text}"

    Return a JSON object with this exact schema:
    {{
        "predicted_stars": <int>,
        "explanation": "<string>"
    }}
    """

# Execution
print("Running Approach 1: Zero-Shot (Groq)...")
df_zero_shot = run_experiment(sampled_df, get_zero_shot_prompt, "Zero-Shot")

# Save
df_zero_shot.to_csv("results_zero_shot_groq.csv", index=False)
print(df_zero_shot)

Running Approach 1: Zero-Shot (Groq)...
--- Starting Experiment: Zero-Shot ---
Processed 50 reviews...
Processed 100 reviews...
Processed 150 reviews...
Processed 200 reviews...
      approach  actual_stars  predicted_stars  \
0    Zero-Shot             4                4   
1    Zero-Shot             5                5   
2    Zero-Shot             3                4   
3    Zero-Shot             1                1   
4    Zero-Shot             5                5   
..         ...           ...              ...   
195  Zero-Shot             4                5   
196  Zero-Shot             4                5   
197  Zero-Shot             3                4   
198  Zero-Shot             4                5   
199  Zero-Shot             5                5   

                                           explanation  json_valid  
0    The reviewer had a generally positive experien...        True  
1    The reviewer expresses high praise for the cra...        True  
2    The reviewer uses pos

In [7]:
def get_few_shot_prompt(review_text):
    return f"""
    Classify the Yelp review into 1-5 stars. Use these examples as a guide:

    Review: "The service was terrible and food cold." -> Output: {{ "predicted_stars": 1, "explanation": "Negative service/food." }}
    Review: "It was okay. Not bad." -> Output: {{ "predicted_stars": 3, "explanation": "Neutral." }}
    Review: "Absolutely amazing! Best pizza ever." -> Output: {{ "predicted_stars": 5, "explanation": "Positive enthusiasm." }}

    Now classify this target Review: "{review_text}"

    Return valid JSON only.
    """

# Execution
print("Running Approach 2: Few-Shot (Groq)...")
df_few_shot = run_experiment(sampled_df, get_few_shot_prompt, "Few-Shot")

# Save
df_few_shot.to_csv("results_few_shot_groq.csv", index=False)
print(df_few_shot)

Running Approach 2: Few-Shot (Groq)...
--- Starting Experiment: Few-Shot ---
Processed 50 reviews...
Processed 100 reviews...
Processed 150 reviews...
Processed 200 reviews...
     approach  actual_stars  predicted_stars  \
0    Few-Shot             4                4   
1    Few-Shot             5                5   
2    Few-Shot             3                4   
3    Few-Shot             1                1   
4    Few-Shot             5                5   
..        ...           ...              ...   
195  Few-Shot             4                5   
196  Few-Shot             4                5   
197  Few-Shot             3                5   
198  Few-Shot             4                5   
199  Few-Shot             5                5   

                                           explanation  json_valid  
0    Positive service/food, with some negative comm...        True  
1                                  Positive enthusiasm        True  
2        Positive experience with food a

In [8]:
def get_cot_prompt(review_text):
    return f"""
    Analyze this Yelp review step-by-step:
    1. Identify positive/negative keywords.
    2. Assess overall tone.
    3. Assign a rating (1-5).

    Review: "{review_text}"

    Return a JSON object with this schema:
    {{
        "predicted_stars": <int>,
        "explanation": "<step-by-step reasoning>"
    }}
    """

# Execution
print("Running Approach 3: Chain-of-Thought (Groq)...")
df_cot = run_experiment(sampled_df, get_cot_prompt, "Chain-of-Thought")

# Save
df_cot.to_csv("results_cot_groq.csv", index=False)
print(df_cot)

Running Approach 3: Chain-of-Thought (Groq)...
--- Starting Experiment: Chain-of-Thought ---
Processed 50 reviews...
Processed 100 reviews...
Processed 150 reviews...
Processed 200 reviews...
             approach  actual_stars  predicted_stars  \
0    Chain-of-Thought             4                4   
1    Chain-of-Thought             5                5   
2    Chain-of-Thought             3                4   
3    Chain-of-Thought             1                1   
4    Chain-of-Thought             5                5   
..                ...           ...              ...   
195  Chain-of-Thought             4                5   
196  Chain-of-Thought             4                5   
197  Chain-of-Thought             3                4   
198  Chain-of-Thought             4                5   
199  Chain-of-Thought             5                5   

                                           explanation  json_valid  
0    Step 1: Positive keywords - 'well made pub gru...        True